# Gas-Water (Air-Water) Generalization Check -- Sister Notebook

Companion to `mixtureTheoryFlow_jax.ipynb`, split off 2026-10-28 to keep
the oil-water and gas-water work separate. See CLAUDE.md's "Reduced-Model
Water-Gas Solver" section for the full reasoning and current status.


In [1]:
# Reduced (alpha_l, Delta) solver -- different state/API than the main
# notebook's solver. See CLAUDE.md's "Reduced-Model Water-Gas Solver".
from mixtureTheoryFlow_jax_solver_water_gas import *


# Carraretto et al. (2020), MDPI -- Zero-Refit Run (Reduced-Model Solver)

Air/water stratified flow, 24m/60mm PMMA pipe, digitized from Figures 6,
9, 10 (`mdpi_airwater_phase2g_dataset.csv`). Horizontal only (`beta=0.0`),
uncalibrated (`f_i_mult=1.0`). See CLAUDE.md's "Reduced-Model Water-Gas
Solver" section for why this dataset motivated the new solver.


## Load the combined MDPI Phase 2g dataset (Figures 6/9/10, digitized)

In [2]:
# Carraretto et al. (2020), digitized Figures 6/9/10. JL/JG are already
# U_sl/U_sg (liquid/gas superficial velocity) -- just renamed below.

import pandas as pd

mdpi_df = pd.read_csv('mdpi_airwater_phase2g_dataset.csv')
mdpi_clean = mdpi_df[['Um_target', 'WC', 'dpdz_measured', 'psi', 'flow_regime', 'JL', 'JG']].copy()
mdpi_clean = mdpi_clean.rename(columns={'JL': 'U_sl', 'JG': 'U_sg'})

print(f"Loaded {len(mdpi_clean)} conditions from Carraretto et al. (2020)")
print(f"U_sl range: {mdpi_clean['U_sl'].min():.4f}-{mdpi_clean['U_sl'].max():.4f} m/s")
print(mdpi_clean.to_string())


Loaded 32 conditions from Carraretto et al. (2020)
U_sl range: 0.0300-0.0600 m/s
    Um_target        WC  dpdz_measured       psi       flow_regime  U_sl      U_sg
0    2.401082  0.012494       4.389125  0.189162  SS(near-2D-wavy)  0.03  2.371082
1    1.888606  0.015885       2.956775  0.192871                SS  0.03  1.858606
2    1.602003  0.018727       2.325055  0.203227                SS  0.03  1.572003
3    1.362869  0.022012       1.687308  0.212411                SS  0.03  1.332869
4    0.957075  0.031346       0.972223  0.216421                SS  0.03  0.927075
5    0.810628  0.037008       0.699853  0.225541                SS  0.03  0.780628
6    0.590077  0.050841       0.413776  0.250026                SS  0.03  0.560077
7    0.450003  0.066666       0.255189  0.274190                SS  0.03  0.420003
8    2.401156  0.016659       4.971498  0.229800  SS(near-2D-wavy)  0.04  2.361156
9    1.888497  0.021181       3.331796  0.245002                SS  0.04  1.848497
10   1

## Validate solver inputs before running


In [ ]:
# D/rho_l/rho_g/mu_l/mu_g/L are fixed constants (next cell) -- only
# U_sl/U_sg need checking here.

required_cols = ['U_sl', 'U_sg']
invalid = mdpi_clean[required_cols].isna().any(axis=1) | mdpi_clean[required_cols].le(0).any(axis=1)
n_bad = int(invalid.sum())
if n_bad:
    print(f"WARNING: {n_bad}/{len(mdpi_clean)} rows have invalid U_sl/U_sg -- see below")
    print(mdpi_clean[invalid])
else:
    print(f"All {len(mdpi_clean)} rows have finite, positive U_sl/U_sg -- safe to run.")


## Run every MDPI condition to steady state, zero-refit

No oscillation sweep/quiet-window step needed -- see CLAUDE.md.


In [3]:
from mixtureTheoryFlow_jax_solver_water_gas import run_real_data_condition
import numpy as np
import pickle

D_mdpi     = 0.060      # pipe diameter [m] -- 60mm ID
L_mdpi     = 24.0       # pipe length [m] -- full 24m PMMA test section
rho_l_mdpi = 998.0      # water [kg/m^3]
rho_g_mdpi = 1.204      # air [kg/m^3] -- 20C, 1 atm
mu_l_mdpi  = 1.0e-3     # water [Pa.s] -- 20C
mu_g_mdpi  = 1.81e-5    # air [Pa.s] -- 20C, 1 atm

mdpi_results = []
for i, row in mdpi_clean.iterrows():
    res = run_real_data_condition(
        row['U_sl'], row['U_sg'], D_mdpi, rho_l_mdpi, rho_g_mdpi, mu_l_mdpi, mu_g_mdpi, L_mdpi
    )
    res['i'] = i
    res['alpha_l_meas'] = row['psi']
    res['dpdz_meas']    = row['dpdz_measured']
    mdpi_results.append(res)
    print(f"[{i:2d}] U_sl={row['U_sl']:.4f}  U_sg={row['U_sg']:.3f}  "
          f"alpha_l_pred={res['alpha_l_pred']:.4f} (meas {row['psi']:.4f})  "
          f"dPdz_l_pred={res['dPdz_pred_liquid']:.2f} (meas {row['dpdz_measured']:.2f})  "
          f"kh={res['kh_total']}  any_nan={res['any_nan']}")

# kh_total>0 means hyperbolicity broke down mid-run (see CLAUDE.md) -- the
# output can look plausible but isn't trustworthy, so exclude before any
# error stat or future calibration fit, same treatment PHI_COLLAPSE got
# in the old solver.
stable = [r for r in mdpi_results if r['kh_total'] == 0]
n_kh_unstable = len(mdpi_results) - len(stable)
if n_kh_unstable:
    print(f"\nWARNING: {n_kh_unstable}/{len(mdpi_results)} conditions had kh_total>0 -- "
          f"excluded from error stats below, not trustworthy.")

alpha_err = np.array([abs(r['alpha_l_pred'] - r['alpha_l_meas']) / r['alpha_l_meas'] for r in stable])
dpdz_err  = np.array([(abs(r['dPdz_pred_liquid']) - r['dpdz_meas']) / r['dpdz_meas'] for r in stable])
print(f"\nMDPI zero-refit (f_i_mult=1.0), {len(stable)}/{len(mdpi_results)} kh-stable: "
      f"holdup mean|err|={alpha_err.mean():.1%}  "
      f"dPdz mean(signed err)={dpdz_err.mean():.1%}  "
      f"any_nan={sum(r['any_nan'] for r in mdpi_results)}/{len(mdpi_results)}")

with open('mdpi_real_data_results.pkl', 'wb') as f:
    pickle.dump(mdpi_results, f)
print("Saved to mdpi_real_data_results.pkl")


[ 0] U_sl=0.0300  U_sg=2.371  alpha_l_pred=0.2406 (meas 0.1892)  dPdz_l_pred=-3.38 (meas 4.39)  kh=0  any_nan=False


KeyboardInterrupt: 

## Fan, Y. (2005) -- Low Liquid Loading Stratified-Wavy (horizontal subset)

`fan2005_sw_subset.csv`, filtered to `Authors=="Fan, Y."`,
`Flow Pattern=="SW"`, 13 horizontal rows (`Inclination angle == 0`) of 60
total -- see CLAUDE.md for why the other 47 stay excluded for now.


### Load Fan, Y. (2005) stratified-wavy (SW) subset

In [4]:
# Fan, Y. (2005), SW subset, horizontal rows only -- beta=0 is the only
# validated regime so far (see CLAUDE.md).

import pandas as pd

fan2005_df = pd.read_csv('fan2005_sw_subset.csv')
fan2005_horiz = fan2005_df[fan2005_df['Inclination angle[°]'] == 0].copy()

fan2005_clean = fan2005_horiz.rename(columns={
    'vSL[m/s]': 'U_sl', 'vSG[m/s]': 'U_sg',
    'rhoL[kg/m3]': 'rho_l', 'rhoG[kg/m3]': 'rho_g',
    'viscosityL[Pa.s]': 'mu_l', 'viscosityG[Pa.s]': 'mu_g',
    'Pipe Diameter[m]': 'D', 'Pipe Length[m]': 'L',
})[['U_sl', 'U_sg', 'D', 'rho_l', 'rho_g', 'mu_l', 'mu_g', 'L']].copy()

fan2005_clean['psi']           = pd.to_numeric(fan2005_horiz['HL[-]'], errors='coerce')  # '-' -> NaN
fan2005_clean['dpdz_measured'] = fan2005_horiz['dP/dL[Pa/m]']
fan2005_clean = fan2005_clean.reset_index(drop=True)

n_no_psi = fan2005_clean['psi'].isna().sum()
print(f"Loaded {len(fan2005_clean)} horizontal SW conditions from Fan, Y. (2005)")
print(f"  {n_no_psi}/{len(fan2005_clean)} have no holdup (HL) measurement -- dP/dL-only")
print(fan2005_clean.to_string())


Loaded 13 horizontal SW conditions from Fan, Y. (2005)
  3/13 have no holdup (HL) measurement -- dP/dL-only
        U_sl   U_sg       D  rho_l  rho_g   mu_l      mu_g   L   psi  dpdz_measured
0   0.018551  15.11  0.0508   1000   2.57  0.001  0.000018  19  0.02         202.08
1   0.014320  19.71  0.0508   1000   2.67  0.001  0.000018  19  0.02         352.72
2   0.013639  15.09  0.0508   1000   2.57  0.001  0.000018  19  0.02         191.78
3   0.010954  15.12  0.0508   1000   2.57  0.001  0.000018  19  0.02         173.70
4   0.010300  19.88  0.0508   1000   2.66  0.001  0.000018  19  0.01         300.24
5   0.007766  14.93  0.0508   1000   2.56  0.001  0.000018  19  0.01         183.17
6   0.006065  19.86  0.0508   1000   2.65  0.001  0.000018  19  0.01         269.21
7   0.004530  14.97  0.0508   1000   2.55  0.001  0.000018  19  0.01         165.56
8   0.003120  20.26  0.0508   1000   2.52  0.001  0.000018  19  0.01         247.31
9   0.002289  14.85  0.0508   1000   2.43  0.001  0.

### Validate solver inputs before running


In [5]:
# This dataset has known placeholder gaps elsewhere (see load cell) --
# check the solver-input columns specifically here.

required_cols = ['U_sl', 'U_sg', 'D', 'rho_l', 'rho_g', 'mu_l', 'mu_g', 'L']
invalid = fan2005_clean[required_cols].isna().any(axis=1) | fan2005_clean[required_cols].le(0).any(axis=1)
n_bad = int(invalid.sum())
if n_bad:
    print(f"WARNING: {n_bad}/{len(fan2005_clean)} rows have invalid solver inputs -- see below")
    print(fan2005_clean[invalid])
else:
    print(f"All {len(fan2005_clean)} rows have finite, positive solver inputs -- safe to run.")


All 13 rows have finite, positive solver inputs -- safe to run.


### Run every horizontal Fan (2005) condition to steady state, zero-refit


In [6]:
from mixtureTheoryFlow_jax_solver_water_gas import run_real_data_condition
import numpy as np
import pickle

fan2005_results = []
for i, row in fan2005_clean.iterrows():
    res = run_real_data_condition(
        row['U_sl'], row['U_sg'], row['D'], row['rho_l'], row['rho_g'],
        row['mu_l'], row['mu_g'], row['L']
    )
    res['i'] = i
    res['alpha_l_meas'] = row['psi']
    res['dpdz_meas']    = row['dpdz_measured']
    fan2005_results.append(res)
    psi_str = f"{row['psi']:.4f}" if pd.notna(row['psi']) else "n/a"
    print(f"[{i:2d}] U_sl={row['U_sl']:.6f}  U_sg={row['U_sg']:.3f}  "
          f"alpha_l_pred={res['alpha_l_pred']:.6f} (meas {psi_str})  "
          f"dPdz_l_pred={res['dPdz_pred_liquid']:.2f} (meas {row['dpdz_measured']:.2f})  "
          f"kh={res['kh_total']}  any_nan={res['any_nan']}")

# kh_total>0 means hyperbolicity broke down mid-run (see CLAUDE.md) -- the
# output can look plausible but isn't trustworthy, so exclude before any
# error stat or future calibration fit, same treatment PHI_COLLAPSE got
# in the old solver.
stable = [r for r in fan2005_results if r['kh_total'] == 0]
n_kh_unstable = len(fan2005_results) - len(stable)
if n_kh_unstable:
    print(f"\nWARNING: {n_kh_unstable}/{len(fan2005_results)} conditions had kh_total>0 -- "
          f"excluded from error stats below, not trustworthy.")

have_psi  = [r for r in stable if pd.notna(r['alpha_l_meas'])]
alpha_err = np.array([abs(r['alpha_l_pred'] - r['alpha_l_meas']) / r['alpha_l_meas'] for r in have_psi])
dpdz_err  = np.array([(abs(r['dPdz_pred_liquid']) - r['dpdz_meas']) / r['dpdz_meas'] for r in stable])

print(f"\nFan (2005) horizontal, zero-refit (f_i_mult=1.0), {len(stable)}/{len(fan2005_results)} kh-stable: "
      f"holdup mean|err|={alpha_err.mean():.1%} ({len(have_psi)}/{len(stable)} have HL)  "
      f"dPdz mean(signed err)={dpdz_err.mean():.1%}  "
      f"any_nan={sum(r['any_nan'] for r in fan2005_results)}/{len(fan2005_results)}")

with open('fan2005_real_data_results.pkl', 'wb') as f:
    pickle.dump(fan2005_results, f)
print("Saved to fan2005_real_data_results.pkl")


[ 0] U_sl=0.018551  U_sg=15.110  alpha_l_pred=0.030719 (meas 0.0200)  dPdz_l_pred=-112.06 (meas 202.08)  kh=0  any_nan=False
[ 1] U_sl=0.014320  U_sg=19.710  alpha_l_pred=0.019094 (meas 0.0200)  dPdz_l_pred=-180.93 (meas 352.72)  kh=15051316  any_nan=False
[ 2] U_sl=0.013639  U_sg=15.090  alpha_l_pred=0.023499 (meas 0.0200)  dPdz_l_pred=-109.74 (meas 191.78)  kh=0  any_nan=False
[ 3] U_sl=0.010954  U_sg=15.120  alpha_l_pred=0.019326 (meas 0.0200)  dPdz_l_pred=-108.97 (meas 173.70)  kh=0  any_nan=False
[ 4] U_sl=0.010300  U_sg=19.880  alpha_l_pred=0.014141 (meas 0.0100)  dPdz_l_pred=-180.96 (meas 300.24)  kh=14890685  any_nan=False
[ 5] U_sl=0.007766  U_sg=14.930  alpha_l_pred=0.014402 (meas 0.0100)  dPdz_l_pred=-104.90 (meas 183.17)  kh=0  any_nan=False
[ 6] U_sl=0.006065  U_sg=19.860  alpha_l_pred=0.008789 (meas 0.0100)  dPdz_l_pred=-177.78 (meas 269.21)  kh=0  any_nan=False
[ 7] U_sl=0.004530  U_sg=14.970  alpha_l_pred=0.008453 (meas 0.0100)  dPdz_l_pred=-234.78 (meas 165.56)  kh=0  

# Vuong (2016), TUFFP -- Oil(Isopar L)-Nitrogen, Zero-Refit Run (Reduced-Model Solver)

Low-liquid-loading oil-gas stratified-wavy flow, 0.156m ID / 85m horizontal
pipe, from the FFPU-FilteredEntrainmentDatabase TUFFP compilation
(`vuong2016_sw_subset.csv`, filtered `Fluids=='Nitrogen - Isopar L'`,
`Flow Pattern=='SW'` -- deliberately excludes the larger `SW&A` (71 rows,
same paper, annular-transition-adjacent) and `ST` (28 rows, two different
inclined-focused papers) labels; see CLAUDE.md for the reasoning).

First fluid pair in this notebook that isn't water-air -- the real
cross-fluid-pair generalization test for `f_i_mult`. Pipe length (85.00m)
sourced from a separate TUFFP parameters file, not present in the FFPU
database itself (see CLAUDE.md). All 36 rows are already horizontal
(`Inclination angle=0`), so no `beta` concerns here.


## Load the Vuong (2016) pure-SW subset

In [ ]:
# Vuong (2016), TUFFP -- pure SW only (excludes SW&A/ST, see title cell).
# rho_g/mu_g are the real measured values at test P/T, no ideal-gas
# back-calc needed. Pipe Length[m]=85.00 was added manually to the CSV
# (sourced from a separate TUFFP parameters file -- see CLAUDE.md).

import pandas as pd

vuong_df = pd.read_csv('vuong2016_sw_subset.csv')
vuong_clean = vuong_df.rename(columns={
    'vSL[m/s]': 'U_sl', 'vSG[m/s]': 'U_sg', 'rhoL[kg/m3]': 'rho_l', 'rhoG[kg/m3]': 'rho_g',
    'viscosityL[Pa.s]': 'mu_l', 'viscosityG[Pa.s]': 'mu_g', 'Pipe Diameter[m]': 'D', 'Pipe Length[m]': 'L',
})[['U_sl', 'U_sg', 'D', 'rho_l', 'rho_g', 'mu_l', 'mu_g', 'L']].copy()
vuong_clean['psi']           = vuong_df['HL[-]']
vuong_clean['dpdz_measured'] = vuong_df['dP/dL[Pa/m]']

print(f"Loaded {len(vuong_clean)} SW conditions from Vuong (2016)")
print(f"U_sl range: {vuong_clean['U_sl'].min():.4f}-{vuong_clean['U_sl'].max():.4f} m/s")
print(vuong_clean.to_string())


## Validate solver inputs before running

In [ ]:
required_cols = ['U_sl', 'U_sg', 'D', 'rho_l', 'rho_g', 'mu_l', 'mu_g', 'L']
invalid = vuong_clean[required_cols].isna().any(axis=1) | vuong_clean[required_cols].le(0).any(axis=1)
n_bad = int(invalid.sum())
if n_bad:
    print(f"WARNING: {n_bad}/{len(vuong_clean)} rows have invalid solver inputs -- see below")
    print(vuong_clean[invalid])
else:
    print(f"All {len(vuong_clean)} rows have finite, positive solver inputs -- safe to run.")


## Run every Vuong (2016) SW condition to steady state, zero-refit

In [ ]:
from mixtureTheoryFlow_jax_solver_water_gas import run_real_data_condition
import numpy as np
import pickle

vuong_results = []
for i, row in vuong_clean.iterrows():
    res = run_real_data_condition(
        row['U_sl'], row['U_sg'], row['D'], row['rho_l'], row['rho_g'],
        row['mu_l'], row['mu_g'], row['L']
    )
    res['i'] = i
    res['alpha_l_meas'] = row['psi']
    res['dpdz_meas']    = row['dpdz_measured']
    vuong_results.append(res)
    print(f"[{i:2d}] U_sl={row['U_sl']:.4f}  U_sg={row['U_sg']:.3f}  "
          f"alpha_l_pred={res['alpha_l_pred']:.4f} (meas {row['psi']:.4f})  "
          f"dPdz_l_pred={res['dPdz_pred_liquid']:.2f} (meas {row['dpdz_measured']:.2f})  "
          f"kh={res['kh_total']}  any_nan={res['any_nan']}")

# kh_total>0 means hyperbolicity broke down mid-run (see CLAUDE.md) -- the
# output can look plausible but isn't trustworthy, so exclude before any
# error stat or future calibration fit, same treatment PHI_COLLAPSE got
# in the old solver.
stable = [r for r in vuong_results if r['kh_total'] == 0]
n_kh_unstable = len(vuong_results) - len(stable)
if n_kh_unstable:
    print(f"\nWARNING: {n_kh_unstable}/{len(vuong_results)} conditions had kh_total>0 -- "
          f"excluded from error stats below, not trustworthy.")

alpha_err = np.array([abs(r['alpha_l_pred'] - r['alpha_l_meas']) / r['alpha_l_meas'] for r in stable])
dpdz_err  = np.array([(abs(r['dPdz_pred_liquid']) - r['dpdz_meas']) / r['dpdz_meas'] for r in stable])
print(f"\nVuong (2016) SW, zero-refit (f_i_mult=1.0), {len(stable)}/{len(vuong_results)} kh-stable: "
      f"holdup mean|err|={alpha_err.mean():.1%}  "
      f"dPdz mean(signed err)={dpdz_err.mean():.1%}  "
      f"any_nan={sum(r['any_nan'] for r in vuong_results)}/{len(vuong_results)}")

with open('vuong2016_real_data_results.pkl', 'wb') as f:
    pickle.dump(vuong_results, f)
print("Saved to vuong2016_real_data_results.pkl")


# Synthetic AD/Gradient Verification

Step 2 of CLAUDE.md's "Reduced-Model Water-Gas Solver" staged plan.
Confirms the optimization machinery can recover a known `f_i_mult` from a
synthetic target -- same purpose Phase 1 served for the old solver's
`drag_coeff`.

Originally attempted via `jax.grad` through `run_real_data_condition_
differentiable`'s full ~60,206-step scan, but that OOM'd this machine
(SIGKILL, exit 137) -- reverse-mode AD keeps every intermediate activation
across the whole trajectory alive for the backward pass, the same wall
the old solver hit at 370,000 steps for `w`. Pivoted to central finite
differences instead, reusing the plain, non-differentiable
`run_real_data_condition` directly: no tape, no `custom_jvp` chain
exercised, memory bounded exactly like a normal forward run. See
CLAUDE.md for the full writeup.


## Generate a synthetic target at a known `f_i_mult`


In [7]:
from mixtureTheoryFlow_jax_solver_water_gas import run_real_data_condition
import numpy as np

# Fan (2005) condition i=0 -- kh_total=0 at f_i_mult=1.0 (fan2005_real_data_results.pkl)
AD_U_sl, AD_U_sg   = 0.018551, 15.110
AD_D, AD_L         = 0.0508, 19.0
AD_rho_l, AD_rho_g = 1000.0, 2.57
AD_mu_l, AD_mu_g   = 1.0e-3, 1.8e-5
AD_N = 150

F_I_MULT_TRUE = 2.0   # deliberately far from the default 1.0

synthetic = run_real_data_condition(
    AD_U_sl, AD_U_sg, AD_D, AD_rho_l, AD_rho_g, AD_mu_l, AD_mu_g, AD_L,
    N=AD_N, f_i_mult=F_I_MULT_TRUE
)
print(synthetic)
assert synthetic['kh_total'] == 0, "synthetic target is kh-unstable -- pick a different f_i_mult_true"
ALPHA_L_TARGET = synthetic['alpha_l_pred']


{'alpha_l_pred': 0.022942834486191555, 'dPdz_pred_liquid': -124.23665797757428, 'dPdz_pred_gas': -124.2366579728095, 'kh_total': 0, 'alpha_min': 0.0012262245075552593, 'alpha_max': 0.02310165985750307, 'dt': 0.0020860390786067847, 'n_steps': 60206, 'any_nan': False}


## Single-point gradient check via central finite differences

No `jax.grad`/tape involved -- two plain forward calls to
`run_real_data_condition` at `f_i_mult +/- h`, same pattern as the old
solver's `w`/`k1`/`k2` fits.


In [8]:
def loss(f_i_mult):
    res = run_real_data_condition(
        AD_U_sl, AD_U_sg, AD_D, AD_rho_l, AD_rho_g, AD_mu_l, AD_mu_g, AD_L,
        N=AD_N, f_i_mult=f_i_mult
    )
    return (res['alpha_l_pred'] - ALPHA_L_TARGET) ** 2

F_I_MULT_GUESS = 1.0
h = 1e-3

fd_grad = (loss(F_I_MULT_GUESS + h) - loss(F_I_MULT_GUESS - h)) / (2 * h)
print(f"central finite-difference grad = {fd_grad:.6f}")

direction = "toward" if np.sign(fd_grad) == np.sign(F_I_MULT_GUESS - F_I_MULT_TRUE) else "away from"
print(f"gradient descent would move f_i_mult {direction} the true value ({F_I_MULT_TRUE})")


central finite-difference grad = -0.000190
gradient descent would move f_i_mult toward the true value (2.0)


## Short recovery optimization (15-epoch Adam, finite-difference gradients)

Only run if the gradient check above looks healthy. Note: with
`learning_rate=0.1` this overshoots past the true value after converging
very close to it around epoch 8 -- a learning-rate tuning artifact (same
symptom the old solver's `k1`/`k2` fit hit before dropping its lr and
adding best-epoch tracking), not evidence the gradient itself is wrong.


In [9]:
import optax

z = np.log(F_I_MULT_GUESS)   # log-space keeps f_i_mult positive, same convention as the old solver's C_D fit
optimizer = optax.adam(learning_rate=0.1)
opt_state = optimizer.init(z)

for epoch in range(15):
    f_i_mult = float(np.exp(z))
    l_center = loss(f_i_mult)
    l_plus   = loss(f_i_mult + h)
    l_minus  = loss(f_i_mult - h)
    d_loss_d_fim = (l_plus - l_minus) / (2 * h)
    # chain rule: d(loss)/dz = d(loss)/d(f_i_mult) * d(f_i_mult)/dz, and
    # d(f_i_mult)/dz = exp(z) = f_i_mult since f_i_mult = exp(z)
    grad_z = d_loss_d_fim * f_i_mult
    updates, opt_state = optimizer.update(grad_z, opt_state)
    z = optax.apply_updates(z, updates)
    print(f"epoch {epoch:2d}  f_i_mult={f_i_mult:.4f}  loss={l_center:.6e}")

print(f"\nRecovered f_i_mult={float(np.exp(z)):.4f}  (true: {F_I_MULT_TRUE})")


epoch  0  f_i_mult=1.0000  loss=6.046787e-05
epoch  1  f_i_mult=1.1052  loss=4.312849e-05
epoch  2  f_i_mult=1.2203  loss=2.910530e-05
epoch  3  f_i_mult=1.3449  loss=1.823957e-05
epoch  4  f_i_mult=1.4782  loss=1.028313e-05
epoch  5  f_i_mult=1.6182  loss=4.904452e-06
epoch  6  f_i_mult=1.7621  loss=1.703875e-06
epoch  7  f_i_mult=1.9063  loss=2.383049e-07
epoch  8  f_i_mult=2.0463  loss=5.292800e-08
epoch  9  f_i_mult=2.1777  loss=7.148021e-07
epoch 10  f_i_mult=2.2958  loss=1.841973e-06
epoch 11  f_i_mult=2.3969  loss=3.123057e-06
epoch 12  f_i_mult=2.4782  loss=4.325360e-06
epoch 13  f_i_mult=2.5379  loss=5.292854e-06
epoch 14  f_i_mult=2.5757  loss=5.937290e-06

Recovered f_i_mult=2.5922  (true: 2.0)
